# V52 - No-chat override action on the frozen champion

Pre-registered hypothesis: the frozen champion gives near-intercept predictions to users with no chat, because its text branch sees an empty row. Replacing the champion rows of NO_CHAT test users with assessment-view predictions improves that slice without touching anyone else. Gates: slice CI95 lower above +0.01 on a FRESH fold seed, global worst-fold non-negative, geometry confined to the slice. Anything else fails close to the untouched champion. Within-user ordering is all NDCG consumes, so cross-user score scales need no calibration.

In [1]:
import json
import re
from hashlib import sha256
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedGroupKFold

SEED = 777
N_SPLITS = 5
SLICE_FLOOR = 0.01
rng = np.random.RandomState(SEED)

CANDIDATES = [Path("/kaggle/input/intelligo-data/data"), Path("data")]
DATA = next((p for p in CANDIDATES if (p / "train_relevance.csv").exists()), None)
assert DATA is not None, "train_relevance.csv not found"
CHAMPION = next((p for p in [Path("/kaggle/input/intelligo-champion/submission_v42_primary.csv"),
                            Path("results/v42/submission_v42_primary.csv")] if p.exists()), None)
assert CHAMPION is not None, "v52 needs the champion file as its base; refusing to invent one"
OUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else "results/v52")
OUT.mkdir(parents=True, exist_ok=True)
print("DATA:", DATA.resolve(), "| CHAMPION:", CHAMPION, "| OUT:", OUT.resolve())

DATA: /home/athilla/Documents/IF_ITB/Lomba/IT-TODAY/intelligo-course-recommender/data | CHAMPION: results/v42/submission_v42_primary.csv | OUT: /home/athilla/Documents/IF_ITB/Lomba/IT-TODAY/intelligo-course-recommender/results/v52


## Views under a fresh fold seed

Assessment and chat Ridge views, same features as v50/v51, fresh seed 777 so the slice measurement is independent of the folds that suggested it.

In [2]:
MODS = [f"M_{i:03d}" for i in range(1, 18)]
train = pd.read_csv(DATA / "train_relevance.csv")
test_ids = pd.read_csv(DATA / "test.csv")["user_id"].tolist()
chats = pd.read_csv(DATA / "chat_history.csv")
assess = pd.read_csv(DATA / "user_assessments.csv")
assert not (set(train["user_id"]) & set(test_ids))
Y = train[MODS].to_numpy(float)
n_train = len(train)

by_user = chats.groupby("user_id")["user_chat_text"].apply(list)
all_users = list(train["user_id"]) + test_ids
is_nochat = np.array([u not in by_user for u in all_users])
nochat_tr = is_nochat[:n_train]
print(f"no-chat: train={nochat_tr.sum()} test={is_nochat[n_train:].sum()}")

blobs = {r["user_id"]: json.loads(r["assessment_result"]) for _, r in assess.iterrows()}
questions = sorted({q for b in blobs.values() for q in b})

def num(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return np.nan

A = np.array([[num(blobs[u].get(q)) for q in questions] for u in all_users])
A = np.where(np.isnan(A), np.nanmean(A, axis=0), A)
A = (A - A.mean(0)) / (A.std(0) + 1e-9)
docs = [" ".join(str(t) for t in by_user.get(u, [])) for u in all_users]
T = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=3).fit_transform(docs)
A_tr, A_te = A[:n_train], A[n_train:]
T_tr, T_te = T[:n_train].toarray(), T[n_train:].toarray()

def template_group(texts):
    if not texts:
        return "NO_CHAT"
    return re.sub(r"\d+", "#", " ".join(str(texts[0]).lower().split()[:6]))
fold_groups = np.array([template_group(by_user.get(u, [])) for u in train["user_id"]])
strata = Y.argmax(1)

def ndcg5(yt, ys):
    order = np.argsort(-ys, axis=1)
    rel = np.take_along_axis(yt, order, 1)[:, :5]
    disc = 1.0 / np.log2(np.arange(2, 7))
    dcg = (rel * disc).sum(1)
    ideal = np.sort(yt, axis=1)[:, ::-1][:, :5]
    idcg = (ideal * disc).sum(1)
    return np.where(idcg > 0, dcg / np.maximum(idcg, 1e-12), 0.0)

def ridge_oof(Xtr, Xte, alpha=3.0):
    pred = np.zeros((Xte.shape[0], len(MODS)))
    oof = np.zeros_like(Y)
    fold_id = np.zeros(len(Xtr), int)
    skf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    for fi, (tr, va) in enumerate(skf.split(Xtr, strata, fold_groups)):
        for j in range(len(MODS)):
            m = Ridge(alpha=alpha, random_state=SEED)
            m.fit(Xtr[tr], Y[tr, j])
            oof[va, j] = m.predict(Xtr[va])
            pred[:, j] += m.predict(Xte) / N_SPLITS
        fold_id[va] = fi + 1
    return oof, pred, fold_id

oof_c, pred_c, fid = ridge_oof(T_tr, T_te)
oof_a, pred_a, _ = ridge_oof(A_tr, A_te)
print(f"chat OOF(seed777): {ndcg5(Y, oof_c).mean():.5f} | assess OOF: {ndcg5(Y, oof_a).mean():.5f}")

no-chat: train=117 test=33


chat OOF(seed777): 0.62249 | assess OOF: 0.43258


## Slice gate, global safety, geometry

Three locks: slice improvement must clear +0.01 at the CI lower bound, no validation fold may go negative globally, and test geometry must stay inside the no-chat slice.

In [3]:
base = ndcg5(Y, oof_c)
act = oof_c.copy()
act[nochat_tr] = oof_a[nochat_tr]
d = ndcg5(Y, act) - base
ds = ndcg5(Y[nochat_tr], act[nochat_tr]) - base[nochat_tr]
lo_s, hi_s = np.quantile([rng.choice(ds, len(ds), replace=True).mean() for _ in range(5000)], [0.025, 0.975])
wf = min(ndcg5(Y[fid == k], act[fid == k]).mean() - base[fid == k].mean() for k in range(1, N_SPLITS + 1))
print(f"pooled delta={d.mean():+.5f} | slice delta={ds.mean():+.5f} CI95=[{lo_s:+.5f},{hi_s:+.5f}] | worst-fold={wf:+.5f}")

ch = pd.read_csv(CHAMPION)
assert ch["user_id"].tolist() == test_ids
C = ch[MODS].to_numpy(float)
S = C.copy()
S[is_nochat[n_train:]] = pred_a[is_nochat[n_train:]]
changed = np.where((S != C).any(1))[0]
test_nochat_idx = set(np.where(is_nochat[n_train:])[0].tolist())
t1 = (C.argmax(1) != S.argmax(1)).sum()
print(f"test changed users={len(changed)} all-in-slice={set(changed.tolist()) <= test_nochat_idx} top1-changes={t1}")

adopt = (lo_s > SLICE_FLOOR) and (wf >= 0) and (set(changed.tolist()) <= test_nochat_idx)
print("ADOPT ACTION:" if adopt else "FAIL-CLOSE: champion untouched", "| slice LoCI", f"{lo_s:+.5f}", "> +0.01" if lo_s > SLICE_FLOOR else "<= +0.01")
final = S if adopt else C
sub = pd.DataFrame(final, columns=MODS)
sub.insert(0, "user_id", test_ids)
sub.to_csv(OUT / "submission_v52_primary.csv", index=False)
print("wrote", OUT / "submission_v52_primary.csv", "sha", sha256(pd.util.hash_pandas_object(sub).values.tobytes()).hexdigest()[:16])

pooled delta=+0.00045 | slice delta=+0.01538 CI95=[-0.01305,+0.04446] | worst-fold=+0.00000
test changed users=33 all-in-slice=True top1-changes=26
FAIL-CLOSE: champion untouched | slice LoCI -0.01305 <= +0.01
wrote results/v52/submission_v52_primary.csv sha 7bc841d13df9f947
